In [1]:
import pandas
from datetime import datetime, timedelta
import json

In [2]:
from Simple_recommender import SimpleRecommender
from risk_recommender import RiskRecommender
from Deontic_recommender import DeonticRecommender
from sncf_recommender import SNCF_RECO, SNCF_RECO1, SNCF_RECO3, SNCF_deontic

In [14]:
# Example JSON inputs

event_payload1 = json.dumps({
"criticality": "HIGH", #MEDIUM or LOW
                    "title": "Passenger fainted",
                    "description": "78 y/o man fainted because of heat",
                    "data": {
                        #"simulation_name" : None,
                        "event_type": "PASSENGER", #INFRASTRUCTURE or IMPACT or HARDWARE
                        "id_train": "1234",
                        "agent_id": "1",
                        "delay": "5 hour",
                        "proba": "0.4"
                    },
                    #"start_date": iso_date,
                    #"end_date": iso_date,
                    "use_case": "Railway"
}
)

event_payload2 = json.dumps({
"criticality": "LOW", #MEDIUM or LOW
                    "title": "Animals on railways",
                    "description": "deers spotted nearby the railway",
                    "data": {
                        #"simulation_name" : None,
                        "event_type": "INFRASTRUCTURE", #or IMPACT or HARDWARE
                        "id_train": "4231",
                        "agent_id": "5",
                        "delay": "2 hour",
                        "proba": "0.8"

                    },
                    #"start_date": iso_date,
                    #"end_date": iso_date,
                    "use_case": "Railway"
}
)

event_payload3 = json.dumps({
"criticality": "MEDIUM", #MEDIUM or LOW
                    "title": "Air conditioner failure",
                    "description": "Air conditioner failure with hot temperature outside",
                    "data": {
                        #"simulation_name" : None,
                        "event_type": "HARDWARE", #or IMPACT or HARDWARE
                        "id_train": "2134",
                        "agent_id": "6",
                        "delay": "1.5 hour",
                        "proba": "0.9"

                    },
                    #"start_date": iso_date,
                    #"end_date": iso_date,
                    "use_case": "Railway"
}
)


context_payload1 = json.dumps({
"data": {
                "trains": { # peut mettre plusieurs trains
                        "id_train": "1234",
                        "nb_passengers_onboard": "97",
                        #"nb_passengers_connection": None,
                        #"latitude": latitude,
                        #"longitude": longitude,
                        #"speed": None,
                        "failure": "FALSE"
                }
                #"list_of_target": None,
                #"direction_agents": None,
                #"position_agents": None,
            },
            #"date": iso_date,
            "use_case": "Railway",
}
)

context_payload2 = json.dumps({
"data": {
                "trains": { # peut mettre plusieurs trains
                        "id_train": "4231",
                        "nb_passengers_onboard": "109",
                        #"nb_passengers_connection": None,
                        #"latitude": latitude,
                        #"longitude": longitude,
                        #"speed": None,
                        "failure": "FALSE"
                }
                #"list_of_target": None,
                #"direction_agents": None,
                #"position_agents": None,
            },
            #"date": iso_date,
            "use_case": "Railway",
}
)


context_payload3 = json.dumps({
"data": {
                "trains": { # peut mettre plusieurs trains
                        "id_train": "2134",
                        "nb_passengers_onboard": "139",
                        #"nb_passengers_connection": None,
                        #"latitude": latitude,
                        #"longitude": longitude,
                        #"speed": None,
                        "failure": "FALSE"
                }
                #"list_of_target": None,
                #"direction_agents": None,
                #"position_agents": None,
            },
            #"date": iso_date,
            "use_case": "Railway",
}
)



In [17]:
type(event_payload1)

str

In [18]:
parsed_recos = [json.loads(r) for r in event_payload1]

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

In [7]:
# Treats events in the same order as they occur (or in the same order as their information is communicated) 
# Passenger -> infrastructure -> hardware
recommender = SimpleRecommender()

pairs = [
    (context_payload1, event_payload1),
    (context_payload2, event_payload2),
    (context_payload3, event_payload3)
]

results = recommender.get_recommendations(pairs)

print(json.dumps(results, indent=2))

[
  [
    {
      "title": "Passenger Malaise",
      "description": "Train 1234 is delayed (5 hour). Recommend informing passengers, adjusting its schedule, and preparing platform change if required.",
      "use_case": "Railway",
      "actions": [
        {
          "action": "notify_passengers"
        },
        {
          "action": "reschedule_train",
          "train_id": "1234"
        },
        {
          "action": "prioritize_track_allocation",
          "train_id": "1234"
        }
      ]
    }
  ],
  [
    {
      "title": "Infrastructure Issue",
      "description": "Infrastructure problem affecting train 4231. Recommend rerouting and adjusting traffic flow.",
      "use_case": "Railway",
      "actions": [
        {
          "action": "reroute_train",
          "train_id": "4231"
        },
        {
          "action": "inspect_track_section"
        },
        {
          "action": "reduce_speed_zone"
        }
      ]
    }
  ],
  [
    {
      "title": "Hardware

In [7]:
results

[[{'title': 'Passenger Malaise',
   'description': 'Train 1234 is delayed (5 hour). Recommend informing passengers, adjusting its schedule, and preparing platform change if required.',
   'use_case': 'Railway',
   'actions': [{'action': 'notify_passengers'},
    {'action': 'reschedule_train', 'train_id': '1234'},
    {'action': 'prioritize_track_allocation', 'train_id': '1234'}]}],
 [{'title': 'Infrastructure Issue',
   'description': 'Infrastructure problem affecting train 4231. Recommend rerouting and adjusting traffic flow.',
   'use_case': 'Railway',
   'actions': [{'action': 'reroute_train', 'train_id': '4231'},
    {'action': 'inspect_track_section'},
    {'action': 'reduce_speed_zone'}]}],
 [{'title': 'Hardware Failure Response',
   'description': 'Hardware issue on train 2134. Recommend stopping the train and sending maintenance.',
   'use_case': 'Railway',
   'actions': [{'action': 'stop_train', 'train_id': '2134'},
    {'action': 'dispatch_maintenance_team'},
    {'action': '

In [8]:
# Treats events by decreasing risk

recommender = RiskRecommender()

pairs = [
    (context_payload1, event_payload1),
    (context_payload2, event_payload2),
    (context_payload3, event_payload3)
]

results = recommender.get_recommendations(pairs)

print(json.dumps(results, indent=2))

total risk estimated = 194.0
total risk estimated = 174.4
total risk estimated = 187.65
[
  {
    "train_id": "1234",
    "event_type": "PASSENGER",
    "title": "Passenger fainted",
    "description": "78 y/o man fainted because of heat",
    "use_case": "Railway",
    "total_risk": 194.0,
    "actions": [
      {
        "action": "provide_medical_assistance"
      },
      {
        "action": "inform_passengers"
      },
      {
        "action": "adjust_schedule"
      }
    ]
  },
  {
    "train_id": "2134",
    "event_type": "HARDWARE",
    "title": "Air conditioner failure",
    "description": "Air conditioner failure with hot temperature outside",
    "use_case": "Railway",
    "total_risk": 187.65,
    "actions": [
      {
        "action": "dispatch_maintenance"
      },
      {
        "action": "assist_passengers"
      },
      {
        "action": "prepare_train_swap"
      }
    ]
  },
  {
    "train_id": "4231",
    "event_type": "INFRASTRUCTURE",
    "title": "Animals o

In [14]:
results

[{'train_id': '1234',
  'event_type': 'PASSENGER',
  'title': 'Passenger fainted',
  'description': '78 y/o man fainted because of heat',
  'use_case': 'Railway',
  'total_risk': 194.0,
  'actions': [{'action': 'provide_medical_assistance'},
   {'action': 'inform_passengers'},
   {'action': 'adjust_schedule'}]},
 {'train_id': '2134',
  'event_type': 'HARDWARE',
  'title': 'Air conditioner failure',
  'description': 'Air conditioner failure with hot temperature outside',
  'use_case': 'Railway',
  'total_risk': 187.65,
  'actions': [{'action': 'dispatch_maintenance'},
   {'action': 'assist_passengers'},
   {'action': 'prepare_train_swap'}]},
 {'train_id': '4231',
  'event_type': 'INFRASTRUCTURE',
  'title': 'Animals on railways',
  'description': 'deers spotted nearby the railway',
  'use_case': 'Railway',
  'total_risk': 174.4,
  'actions': [{'action': 'reduce_speed'},
   {'action': 'inspect_track'},
   {'action': 'reroute_if_needed'}]}]

In [9]:
# Treats events by respecting rules (here : thresholds must not be exceeded)

deontic_recommender = DeonticRecommender()

pairs = [
    (context_payload1, event_payload1),
    (context_payload2, event_payload2),
    (context_payload3, event_payload3)
]

results = deontic_recommender.get_recommendations(
    pairs,
    type="delay",
    delay_threshold="3"
)

print(json.dumps(results, indent=2))


[
  {
    "train_id": "1234",
    "event_type": "PASSENGER",
    "title": "Passenger fainted",
    "description": "78 y/o man fainted because of heat",
    "delay_hours": 5.0,
    "passengers_onboard": 97,
    "severity_score": 485.0,
    "rule_triggered": true,
    "rule_type": "delay",
    "actions": [
      {
        "action": "provide_medical_assistance"
      },
      {
        "action": "inform_passengers"
      }
    ]
  },
  {
    "train_id": "4231",
    "event_type": "INFRASTRUCTURE",
    "title": "Animals on railways",
    "description": "deers spotted nearby the railway",
    "delay_hours": 2.0,
    "passengers_onboard": 109,
    "severity_score": 218.0,
    "rule_triggered": false,
    "rule_type": "delay",
    "actions": [
      {
        "action": "reduce_speed"
      },
      {
        "action": "inspect_track"
      }
    ]
  },
  {
    "train_id": "2134",
    "event_type": "HARDWARE",
    "title": "Air conditioner failure",
    "description": "Air conditioner failure w

In [11]:
deontic_recommender = DeonticRecommender()

pairs = [
    (context_payload1, event_payload1),
    (context_payload2, event_payload2),
    (context_payload3, event_payload3)
]

results = deontic_recommender.get_recommendations(
    pairs,
    type="passengers",
    passengers_threshold="120"
)

print(json.dumps(results, indent=2))

[
  {
    "train_id": "2134",
    "event_type": "HARDWARE",
    "title": "Air conditioner failure",
    "description": "Air conditioner failure with hot temperature outside",
    "delay_hours": 1.5,
    "passengers_onboard": 139,
    "severity_score": 208.5,
    "rule_triggered": true,
    "rule_type": "passengers",
    "actions": [
      {
        "action": "dispatch_maintenance"
      },
      {
        "action": "assist_passengers"
      }
    ]
  },
  {
    "train_id": "1234",
    "event_type": "PASSENGER",
    "title": "Passenger fainted",
    "description": "78 y/o man fainted because of heat",
    "delay_hours": 5.0,
    "passengers_onboard": 97,
    "severity_score": 485.0,
    "rule_triggered": false,
    "rule_type": "passengers",
    "actions": [
      {
        "action": "provide_medical_assistance"
      },
      {
        "action": "inform_passengers"
      }
    ]
  },
  {
    "train_id": "4231",
    "event_type": "INFRASTRUCTURE",
    "title": "Animals on railways",
   

In [12]:
deontic_recommender = DeonticRecommender()

pairs = [
    (context_payload1, event_payload1),
    (context_payload2, event_payload2),
    (context_payload3, event_payload3)
]

results = deontic_recommender.get_recommendations(
    pairs,
    type="both",
    passengers_threshold="100",
    delay_threshold="2"
    
)

print(json.dumps(results, indent=2))

[
  {
    "train_id": "1234",
    "event_type": "PASSENGER",
    "title": "Passenger fainted",
    "description": "78 y/o man fainted because of heat",
    "delay_hours": 5.0,
    "passengers_onboard": 97,
    "severity_score": 485.0,
    "rule_triggered": false,
    "rule_type": "both",
    "actions": [
      {
        "action": "provide_medical_assistance"
      },
      {
        "action": "inform_passengers"
      }
    ]
  },
  {
    "train_id": "4231",
    "event_type": "INFRASTRUCTURE",
    "title": "Animals on railways",
    "description": "deers spotted nearby the railway",
    "delay_hours": 2.0,
    "passengers_onboard": 109,
    "severity_score": 218.0,
    "rule_triggered": false,
    "rule_type": "both",
    "actions": [
      {
        "action": "reduce_speed"
      },
      {
        "action": "inspect_track"
      }
    ]
  },
  {
    "train_id": "2134",
    "event_type": "HARDWARE",
    "title": "Air conditioner failure",
    "description": "Air conditioner failure wi

## Partie basée sur le tableau UC SNCF

In [3]:
event_payload1 = json.dumps({
"criticality": "HIGH", #MEDIUM or LOW
                    "title": "Malaise Voyageurs, Dérangement d'installation, Gare de Poitiers",
                    "description": "Malaise Voyageurs au TGV AB001. Intervention des pompiers en gare de Poitiers. Au même moment, dérangement d'installation à Poitiers accès impossible. Heure de restitution prévisible: 12h00." ,
                    "data": {
                        #"simulation_name" : None,
                        "id_event":"1",
                        "event_type": "PASSENGER", #INFRASTRUCTURE or IMPACT or HARDWARE
                        "id_train": "AB001",
                        "agent_id": "1",
                        "delay": 5,
                        #"proba": "0.4"
                    },
                    #"start_date": (datetime.now() + timedelta(seconds=10)),
                    #"end_date": (start_date + timedelta(hours=2)).isoformat(),
                    "use_case": "Railway"
}
)


context_payload1 = json.dumps({
"data": {
                "trains": { # peut mettre plusieurs trains
                        "id_train": "AB001",
                        "nb_passengers_onboard": 468,
                        "trajet" : "Paris/Bordeaux",
                        "arrets":"Poitiers/Bordeaux",
                        #"nb_passengers_connection": None,
                        "latitude": 46.583328,
                        "longitude": 0.33333,
                        #"speed": None,
                        "failure": True
                }
        }       
}         
)

In [4]:
event_payload2 = json.dumps({
#"criticality": "HIGH", #MEDIUM or LOW
                    "title": "Caténaire arrachée ",
                    "description": "Le TGV CD001 a arraché la caténaire au niveau du poste 60 de Pliboux sur la LGV SEA. La voie est contigüe n'est pas accessible. Les trains sont détournés par la ligne classique entre Poitiers et  Angoulême. Heure de restitution prévisible d'une voie : 23h00.",
                    "data": {
                        #"simulation_name" : None,
                        "id_event":"2",
                        "event_type": "INFRASTRUCTURE", #INFRASTRUCTURE or IMPACT or HARDWARE
                       "id_train": "AB001",
                        "latitude": 46.167,
                        "longitude": 0.127,
                        "agent_id": "1",
                        "delay": 5,
                        #"proba": "0.4"
                    },
                    #"start_date": (datetime.now() + timedelta(seconds=30)).isoformat(),
                    #"end_date": (start_date + timedelta(hours=4)).isoformat(),
                    "use_case": "Railway"
}
)


context_payload2 = json.dumps({
"data": {
                "trains": { # peut mettre plusieurs trains
                        "id_train": "AB001",
                        "nb_passengers_onboard": 459,
                        "trajet" : "Paris/Bordeaux",
                        "arrets":"Angoulême/Bordeaux",
                        #"nb_passengers_connection": None,
                        #"latitude": latitude,
                        #"longitude": longitude,
                        #"speed": None,
                        "failure": False
                }
        }
}
)

In [5]:
event_payload3 = json.dumps({
"criticality": "HIGH", #MEDIUM or LOW
                    "title": "Fuite de gaz de ville",
                    "description": "Fuite de gaz au niveau de Bassens. La plateforme ferroviaire se retrouve dans le périmètre de sécurité. Les autorités décident d'interdir la circulation. Heure de restitution prévisible : 19h00.",
                    "data": {
                        #"simulation_name" : None,
                        "event_type": "INFRASTRUCTURE", #INFRASTRUCTURE or IMPACT or HARDWARE
                        "id_event":"3",
                        "id_train": "EF004",
                        "latitude" : 44.900002,
                        "longitude" : -0.51667,

                        #"agent_id": "1",
                        "delay": 5,
                        #"proba": "0.4"
                    },
                    #"start_date": (datetime.now() + timedelta(seconds=40)),
                    #"end_date": (start_date + timedelta(hours=5)).isoformat(),
                    "use_case": "Railway"
}
)

context_payload3 = json.dumps({
"data": {
                "trains": [{ # peut mettre plusieurs trains
                        "id_train": "EF004",
                        "nb_passengers_onboard": 348,
                        "trajet" : "Bordeaux / Paris",
                        "arrets":"Bordeaux / Libourne/ Angoulême / Poitiers / Chatellerault",
                        #"nb_passengers_connection": None,
                        #"latitude": latitude,
                        #"longitude": longitude,
                        #"speed": None,
                        "failure": False
                }]
        }
}
)

#### English context/events


In [3]:
start_date = datetime.now() + timedelta(seconds=10)

event_payload1 = json.dumps({
"criticality": "HIGH", #MEDIUM or LOW
                    "title": "Passenger taken ill in Poitiers",
                    "description": "Passenger taken ill in TGV AB001. emergency services intervention in the Poitiers station. Impossible to access the Poitiers station. Estimated time of service traffic resumption : 12:00." ,
                    "data": {
                        #"simulation_name" : None,
                        "id_event": "1",
                        "event_type": "PASSENGER", #INFRASTRUCTURE or IMPACT or HARDWARE
                        "id_train": "AB001",
                        "agent_id": "1",
                        "delay": 0,
                        #"proba": "0.4"
                    },
                    "start_date": (start_date).isoformat(),
                    "end_date": (start_date + timedelta(hours=2)).isoformat(),
                    "use_case": "Railway"
}
)


context_payload1 = json.dumps({
"data": {
                "trains":[{ # peut mettre plusieurs trains
                        "id_train": "AB001",
                        "nb_passengers_onboard": 468,
                        "trip" : "Paris/Bordeaux",
                        "stops":"Poitiers/Bordeaux",
                        #"nb_passengers_connection": 0,
                        "latitude": 46.583328,
                        "longitude": 0.33333,
                        #"speed": None,
                        "failure": True
                }]
        },  
"use_case": "Railway"     
}         
)




# context_payload1 = json.dumps({
# "data": {
#                 "trains": { # peut mettre plusieurs trains
#                         "id_train": "1234",
#                         #"nb_passengers_onboard": None,
#                         #"nb_passengers_connection": None,
#                         #"latitude": latitude,
#                         #"longitude": longitude,
#                         #"speed": None,
#                         "failure": "FALSE"
#                 }
#                 #"list_of_target": None,
#                 #"direction_agents": None,
#                 #"position_agents": None,
#             },
#             #"date": iso_date,
#             "use_case": "Railway",
# }
# )



event_payload2 = json.dumps({
"criticality": "HIGH", #MEDIUM or LOW
                    "title": "Broken overhead wire",
                    "description": "Broken overhead wire for TGV CD001 in the Pliboux plant on line LGV SEA. The adjacent track is not accessible. Trains are rerouted to the standard line between Poitiers and Angoulême. Estimated time of service traffic resumption : 23:00",
                    "data": {
                        #"simulation_name" : None,
                        "id_event":"2",
                        "event_type": "INFRASTRUCTURE", #INFRASTRUCTURE or IMPACT or HARDWARE
                       "id_train": "AB001",
                        "latitude": "46.167",
                        "longitude": "0.127",
                        "agent_id": "1",
                        "delay":  0,
                        #"proba": "0.4"
                    },
                    "start_date": (start_date + timedelta(seconds=30)).isoformat(),
                    "end_date": (start_date + timedelta(hours=4)).isoformat(),
                    "use_case": "Railway"
}
)


context_payload2 = json.dumps({
"data": {
                "trains": [{ # peut mettre plusieurs trains
                        "id_train": "AB001",
                        "nb_passengers_onboard": "459",
                        "trip" : "Paris/Bordeaux",
                        "stops":"Angoulême/Bordeaux",
                        #"nb_passengers_connection": None,
                        #"latitude": latitude,
                        #"longitude": longitude,
                        #"speed": None,
                        "failure": False
                }]
        },  
"use_case": "Railway"     
}
)


event_payload3 = json.dumps({
"criticality": "HIGH", #MEDIUM or LOW
                    "title": "Gas leak",
                    "description": "Gas leak in the city of Bassens. The railway facility is in the scurity perimeter. Circulation is forbidden by authorities. Estimated time of traffic resumption : 19:00.",
                    "data": {
                        #"simulation_name" : None,
                        "event_type": "INFRASTRUCTURE", #INFRASTRUCTURE or IMPACT or HARDWARE
                        "id_event":"3",
                        "id_train": "EF004",
                        "latitude" : "44.900002",
                        "longitude" : "-0.51667 ",
                        "agent_id" : "1",
                        "delay": 0,
                        #"proba": "0.4"
                    },
                    "start_date": (start_date + timedelta(seconds=40)).isoformat(),
                    "end_date": (start_date + timedelta(hours=5)).isoformat(),
                    "use_case": "Railway"
}
)

context_payload3 = json.dumps({
"data": {
                "trains": [{ # peut mettre plusieurs trains
                        "id_train": "EF004",
                        "nb_passengers_onboard": "348",
                        "trip" : "Bordeaux / Paris",
                        "stops":"Bordeaux / Libourne/ Angoulême / Poitiers / Chatellerault",
                        #"nb_passengers_connection": None,
                        #"latitude": latitude,
                        #"longitude": longitude,
                        #"speed": None,
                        "failure": False
                }]
        },  
"use_case": "Railway"     
}
)


#### Reco json


In [4]:
{  "title": "Title displayed on each reco",
  "description": "Detailed description to help operator understand the proposed reco",
  "use_case": "Railway",
  "agent_type": "AI",
  "kpis": {
    "kpi1_name": "kpi1_value",
    "kpi2_name": "kpi2_value"}}


{'title': 'Title displayed on each reco',
 'description': 'Detailed description to help operator understand the proposed reco',
 'use_case': 'Railway',
 'agent_type': 'AI',
 'kpis': {'kpi1_name': 'kpi1_value', 'kpi2_name': 'kpi2_value'}}

In [5]:
Reco11 = json.dumps({
"data": {
                "title": "Stay at the St Pierre des Corps station",
                "description" : "Hold the train in the St Pierre des Corps station until traffic resumes.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "78",
                        "nb_impacted_passengers" : "468",
                        "total_cost" : "32838",
                        "delay" : "1h30",
                        "best": "True",
                        }
                    
        }
}
)

Reco12 = json.dumps({
"data": {
                "title": "Delay",
                "description" : "Hold the train on the track until traffic resumes. Arrangements for passenger support.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "87",
                        "nb_impacted_passengers" : "398",
                        "total_cost" : "34626",
                        "delay" : "1h30",
                        "best": "False",
                        }
                    
        }
}
)


Reco13 = json.dumps({
"data": {
                "title": "Reroute",
                "description" : "Reroute the trains. Passengers traveling to and from Poitiers are being assisted.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "102",
                        "nb_impacted_passengers" : "350",
                        "total_cost" : "35700",
                        "delay" : "1h40",
                        "best": "False",
                        }
                    
        }
}
)


Reco14 = json.dumps({
"data": {
                "title": "Cancel",
                "description" : "Train services are cancelled. Passengers are advised to postpone their journey.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "233",
                        "nb_impacted_passengers" : "468",
                        "total_cost" : "109044",
                        "delay" : "1h45",
                        "best": "False",
                        }
                    
        }
}
)



In [6]:

Reco21 = json.dumps({
"data": {
                "title": "Stay at the station",
                "description" : "Hold the train in the Poitiers station until traffic resumes.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "123",
                        "nb_impacted_passengers" : "459",
                        "total_cost" : "56457",
                        "delay" : "1h05",
                        "best": "False",
                        }
                    
        }
}
)





Reco22 = json.dumps({
"data": {
                "title": "Delay",
                "description" : "Hold the train on the track until traffic resumes. Arrangements for passenger support.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "129",
                        "nb_impacted_passengers" : "459",
                        "total_cost" : "59211",
                        "delay" : "1h00",
                        "best": "False",
                        }
                    
        }
}
)

Reco23 = json.dumps({
"data": {
                "title": "Reroute",
                "description" : "Diversion and rerouting of trains to the standard line between Angoulême and Poitiers. ",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "73",
                        "nb_impacted_passengers" : "459",
                        "total_cost" : "33507",
                        "delay" : "1h00",
                        "best": "True",
                        }
                    
        }
}
)


Reco24 = json.dumps({
"data": {
                "title": "Cancel",
                "description" : "Train services are cancelled. Passengers are advised to postpone their journey.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "221",
                        "nb_impacted_passengers" : "459",
                        "total_cost" : "101439",
                        "delay" : "1h00",
                        "best": "False",
                        }
                    
        }
}
)



In [7]:
Reco31 = json.dumps({
"data": {
                "title": "Wait at the Bordeaux station",
                "description" : "Hold the train at the station until traffic resumes.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "94",
                        "nb_impacted_passengers" : "348",
                        "total_cost" : "32712",
                        "delay" : "00h30",
                        "best": "True",
                        }
                    
        }
}
)


Reco32 = json.dumps({
"data": {
                "title": "Delay",
                "description" : "Hold the train on the track until traffic resumes. Arrangements for passenger support.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "94",
                        "nb_impacted_passengers" : "347",
                        "total_cost" : "32618",
                        "delay" : "00h53",
                        "best": "False",
                        }
                    
        }
}
)


Reco33 = json.dumps({
"data": {
                "title": "Rerouting from Bordeaux to Angoulême ",
                "description" : "Diversion and rerouting of trains to the standard line between Angoulême and Poitiers.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "95",
                        "nb_impacted_passengers" : "348",
                        "total_cost" : "33060",
                        "delay" : "1h24",
                        "best": "False",
                        }
                    
        }
}
)

Reco34 = json.dumps({
"data": {
                "title": "Supprimer",
                "description" : "Train services are cancelled. Passengers are advised to postpone their journey.",
                "use_case": "Railway",
                "agent_type": "AI",
                "kpis": {
                        "cost" : "257",
                        "nb_impacted_passengers" : "348",
                        "total_cost" : "89436",
                        "delay" : "00h40",
                        "best": "False",
                        }
                    
        }
}
)




In [8]:
RECOMMENDATION_CATALOG = {
    "1": [Reco11, Reco12, Reco13, Reco14],
    "2": [Reco21, Reco22, Reco23, Reco24],
    "3": [Reco31, Reco32, Reco33, Reco34],
}


In [9]:
recommendations = SNCF_RECO3(event_payload3, context_payload3, RECOMMENDATION_CATALOG)

for r in recommendations:
    reco = json.loads(r)
    print(reco["data"]["title"], "|", reco["data"]["description"],"| KPIs:", reco["data"]["kpis"])


Wait at the Bordeaux station | Hold the train at the station until traffic resumes. | KPIs: {'cost': '94', 'nb_impacted_passengers': '348', 'total_cost': '32712', 'delay': '00h30', 'best': 'True'}
Delay | Hold the train on the track until traffic resumes. Arrangements for passenger support. | KPIs: {'cost': '94', 'nb_impacted_passengers': '347', 'total_cost': '32618', 'delay': '00h53', 'best': 'False'}
Rerouting from Bordeaux to Angoulême  | Diversion and rerouting of trains to the standard line between Angoulême and Poitiers. | KPIs: {'cost': '95', 'nb_impacted_passengers': '348', 'total_cost': '33060', 'delay': '1h24', 'best': 'False'}
Supprimer | Train services are cancelled. Passengers are advised to postpone their journey. | KPIs: {'cost': '257', 'nb_impacted_passengers': '348', 'total_cost': '89436', 'delay': '00h40', 'best': 'False'}


In [10]:
recommendations = SNCF_deontic(
    event_payload3,
    context_payload3,
    RECOMMENDATION_CATALOG,
    type="delay",
    threshold_value="1h30"
)

In [11]:
for r in recommendations:
    reco = json.loads(r)
    print(reco["data"]["title"], "|", reco["data"]["description"],"| KPIs:", reco["data"]["kpis"])

Wait at the Bordeaux station | Hold the train at the station until traffic resumes. | KPIs: {'cost': '94', 'nb_impacted_passengers': '348', 'total_cost': '32712', 'delay': '00h30', 'best': 'True'}
Supprimer | Train services are cancelled. Passengers are advised to postpone their journey. | KPIs: {'cost': '257', 'nb_impacted_passengers': '348', 'total_cost': '89436', 'delay': '00h40', 'best': 'False'}
Delay | Hold the train on the track until traffic resumes. Arrangements for passenger support. | KPIs: {'cost': '94', 'nb_impacted_passengers': '347', 'total_cost': '32618', 'delay': '00h53', 'best': 'False'}
Rerouting from Bordeaux to Angoulême  | Diversion and rerouting of trains to the standard line between Angoulême and Poitiers. | KPIs: {'cost': '95', 'nb_impacted_passengers': '348', 'total_cost': '33060', 'delay': '1h24', 'best': 'False'}


In [12]:
recommendations = SNCF_deontic(
    event_payload3,
    context_payload3,
    RECOMMENDATION_CATALOG,
    type="passengers",
    threshold_value=350
)

In [13]:
for r in recommendations:
    reco = json.loads(r)
    print(reco["data"]["title"], "|", reco["data"]["description"],"| KPIs:", reco["data"]["kpis"])

Delay | Hold the train on the track until traffic resumes. Arrangements for passenger support. | KPIs: {'cost': '94', 'nb_impacted_passengers': '347', 'total_cost': '32618', 'delay': '00h53', 'best': 'False'}
Wait at the Bordeaux station | Hold the train at the station until traffic resumes. | KPIs: {'cost': '94', 'nb_impacted_passengers': '348', 'total_cost': '32712', 'delay': '00h30', 'best': 'True'}
Rerouting from Bordeaux to Angoulême  | Diversion and rerouting of trains to the standard line between Angoulême and Poitiers. | KPIs: {'cost': '95', 'nb_impacted_passengers': '348', 'total_cost': '33060', 'delay': '1h24', 'best': 'False'}
Supprimer | Train services are cancelled. Passengers are advised to postpone their journey. | KPIs: {'cost': '257', 'nb_impacted_passengers': '348', 'total_cost': '89436', 'delay': '00h40', 'best': 'False'}


In [18]:
recommendations = SNCF_deontic(
    event_payload3,
    context_payload3,
    RECOMMENDATION_CATALOG,
    type="cost",
    threshold_value=300
)

In [19]:
for r in recommendations:
    reco = json.loads(r)
    print(reco["data"]["title"], "|", reco["data"]["description"],"| KPIs:", reco["data"]["kpis"])

Wait at the Bordeaux station | Hold the train at the station until traffic resumes. | KPIs: {'cost': '94', 'nb_impacted_passengers': '348', 'total_cost': '32712', 'delay': '00h30', 'best': 'True'}
Delay | Hold the train on the track until traffic resumes. Arrangements for passenger support. | KPIs: {'cost': '94', 'nb_impacted_passengers': '347', 'total_cost': '32618', 'delay': '00h53', 'best': 'False'}
Rerouting from Bordeaux to Angoulême  | Diversion and rerouting of trains to the standard line between Angoulême and Poitiers. | KPIs: {'cost': '95', 'nb_impacted_passengers': '348', 'total_cost': '33060', 'delay': '1h24', 'best': 'False'}
Supprimer | Train services are cancelled. Passengers are advised to postpone their journey. | KPIs: {'cost': '257', 'nb_impacted_passengers': '348', 'total_cost': '89436', 'delay': '00h40', 'best': 'False'}


In [28]:
recommendations = SNCF_deontic(
    event_payload3,
    context_payload3,
    RECOMMENDATION_CATALOG,
    type="total_cost",
    threshold_value=100000
)

In [29]:
for r in recommendations:
    reco = json.loads(r)
    print(reco["data"]["title"], "|", reco["data"]["description"],"| KPIs:", reco["data"]["kpis"])

Delay | Hold the train on the track until traffic resumes. Arrangements for passenger support. | KPIs: {'cost': '94', 'nb_impacted_passengers': '347', 'total_cost': '32618', 'delay': '00h53', 'best': 'False'}
Wait at the Bordeaux station | Hold the train at the station until traffic resumes. | KPIs: {'cost': '94', 'nb_impacted_passengers': '348', 'total_cost': '32712', 'delay': '00h30', 'best': 'True'}
Rerouting from Bordeaux to Angoulême  | Diversion and rerouting of trains to the standard line between Angoulême and Poitiers. | KPIs: {'cost': '95', 'nb_impacted_passengers': '348', 'total_cost': '33060', 'delay': '1h24', 'best': 'False'}
Supprimer | Train services are cancelled. Passengers are advised to postpone their journey. | KPIs: {'cost': '257', 'nb_impacted_passengers': '348', 'total_cost': '89436', 'delay': '00h40', 'best': 'False'}
